# Per-gene TWAS Z + Mendelian Randomization

## Description

Fan out per row of a per-gene manifest. Each row pairs a gene's `TwasWeights` RDS with the matching `GwasSumStats` RDS for that gene's home LD block; the SoS step runs `twas.R` per row, which calls `pecotmr::causalInferencePipeline()` to compute the gene's TWAS Z + (optional) MR statistics. Optional per-gene `FineMappingResult` is wired through too.

## Inputs

- `--manifest` &mdash; per-gene manifest TSV with columns:
    - `gene_id` &mdash; gene identifier (unique; used in the output filename).
    - `twas_weights_rds` &mdash; path to the per-gene `TwasWeights` RDS (single TwasWeightsEntry).
    - `gwas_sumstats_rds` &mdash; path to the per-LD-block `GwasSumStats` RDS that covers the gene's home block.
    - `fine_mapping_result_rds` &mdash; optional; path to the matching per-gene `FineMappingResult` RDS (leave blank to skip).
- `--study` &mdash; study identifier (used in output filenames).
- `--mr-pip-cutoff` &mdash; pass-through (default 0.5).
- `--mr-method` &mdash; pass-through (`"ivwPerVariant"` or `"csAware"`; default `"ivwPerVariant"`).

## Output

- `{cwd}/{study}.{gene_id}.twas.rds` per gene.


## Example

```bash
sos run pipeline/twas.ipynb twas \
    --cwd output --modular-script-dir /path/to/code/script \
    --study protocol_example_chr22 \
    --manifest /tmp/gwas_smoke/twas_manifest.tsv
```

Manifest example:
```
gene_id	twas_weights_rds	gwas_sumstats_rds	fine_mapping_result_rds
ENSG00000130538	/tmp/gwas_smoke/blessed_tw.rds	/tmp/gwas_smoke/blocks/chr22_10516173_17414263.rds	
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: manifest = path
parameter: mr_pip_cutoff = 0.5
parameter: mr_method = 'ivwPerVariant'
parameter: modular_script_dir = path('code/script')
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '8G'
parameter: numThreads = 1


In [ ]:
[validate_manifest]
# Validate + canonicalise the user-supplied per-gene TWAS manifest via
# twas_manifest_validate.R. Downstream fans out over its rows; no
# Python parsing in this notebook.
input: manifest
output: f"{cwd}/{study}.twas_manifest.tsv"
task: trunk_workers = 1, trunk_size = 1, walltime = '15m', mem = '2G', cores = 1, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/twas_manifest_validate.R \
        --manifest ${_input} \
        --output ${_output}

In [ ]:
[twas]
# Fan out over the canonical manifest's rows: one task per gene_id.
# Manifest columns: gene_id, twas_weights_rds, gwas_sumstats_rds, and
# optionally fine_mapping_result_rds.
import csv
jobs = list(csv.DictReader(open(f"{cwd}/{study}.twas_manifest.tsv"), delimiter='\t'))
input: for_each = 'jobs'
output: f"{cwd}/{study}.{_jobs['gene_id']}.twas.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/twas.R \
        --twas-weights ${_jobs['twas_weights_rds']} \
        --gwas-sumstats ${_jobs['gwas_sumstats_rds']} \
        ${('--fine-mapping-result ' + _jobs.get('fine_mapping_result_rds', '')) if _jobs.get('fine_mapping_result_rds') else ''} \
        --mr-pip-cutoff ${mr_pip_cutoff} \
        --mr-method ${mr_method} \
        --output ${_output}